# Media Pipeline Calibration & Benchmarking

This notebook traverses the platform-specific calibration media items under `backend/tests/calib_test` and evaluates them using both AIGC classifier backends: 
1. **SigLIP** (`Ateeqq/ai-vs-human-image-detector`)
2. **ConvNeXt** (`xRayon/convnext-ai-images-detector` via `timm`)

It evaluates face deepfakes using **GenD** (`yermandy/GenD_CLIP_L_14`) with **YuNet** face-detection preprocessing.

The purpose is to analyze platform-specific differences (Facebook vs Instagram vs TikTok) and inspect classification errors to identify model weaknesses.

In [ ]:
import os
import sys
from pathlib import Path

# Add parent directories to path so we can import gend from backend/models
current_dir = Path(os.getcwd()).resolve()
project_root = current_dir.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

backend_dir = project_root / "backend"
if str(backend_dir) not in sys.path:
    sys.path.insert(0, str(backend_dir))

models_dir = backend_dir / "models"
if str(models_dir) not in sys.path:
    sys.path.insert(0, str(models_dir))

print("Project Root:", project_root)
print("Backend Dir:", backend_dir)
print("Models Dir:", models_dir)

# Optional: Un-comment to install packages if needed
# !pip install -q opencv-python-headless transformers torch torchvision pillow matplotlib pandas safetensors huggingface_hub timm

import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd
import torchvision.transforms as T

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active device: {device}")

In [ ]:
# YuNet face detector setup (backed by Hugging Face mirror copy to resolve LFS stubs)
import shutil
from huggingface_hub import hf_hub_download

YUNET_FILENAME = "face_detection_yunet_2023mar.onnx"
YUNET_MODEL_PATH = YUNET_FILENAME

def _looks_like_onnx(path, min_bytes=100_000):
    if not os.path.exists(path):
        return False
    if os.path.getsize(path) < min_bytes:
        return False
    with open(path, "rb") as f:
        head = f.read(20)
    if head.strip().startswith(b"<") or head.strip().startswith(b"version"):
        return False
    return True

if _looks_like_onnx(YUNET_MODEL_PATH):
    print(f"Found valid YuNet model at {YUNET_MODEL_PATH}")
else:
    if os.path.exists(YUNET_MODEL_PATH):
        os.remove(YUNET_MODEL_PATH)
    print("Downloading YuNet model from Hugging Face...")
    try:
        downloaded_path = hf_hub_download(
            repo_id="opencv/face_detection_yunet",
            filename=YUNET_FILENAME,
        )
        shutil.copy(downloaded_path, YUNET_MODEL_PATH)
        print("Download complete and validated.")
    except Exception as e:
        print(f"Download failed: {e}")

# Initialize YuNet detector instance
face_detector = cv2.FaceDetectorYN.create(
    model=YUNET_MODEL_PATH,
    config="",
    input_size=(320, 320),
    score_threshold=0.6,
    nms_threshold=0.3,
    top_k=5000,
)
print("YuNet face detector initialized successfully!")

In [ ]:
# Load Deepfake detector (GenD) and AIGC Classifiers (Ateeqq & ConvNeXt)
from transformers import pipeline
from safetensors.torch import load_model
import timm

# Import configuration and GenD model class from backend modeling
from gend import GenDConfig, GenD

# 1. GenD deepfake detector
print("Loading GenD CLIP model...")
weights_path = hf_hub_download(repo_id="yermandy/GenD_CLIP_L_14", filename="model.safetensors")
config = GenDConfig(backbone="openai/clip-vit-large-patch14", head="LinearNorm")
deepfake_model = GenD(config)
load_model(deepfake_model, weights_path)
deepfake_model.eval()
deepfake_model.to(device)
print("GenD model successfully loaded!")

# 2. SigLIP AIGC model (Ateeqq/ai-vs-human-image-detector)
print("Loading Ateeqq AI-vs-Human Detector pipeline...")
aigc_pipeline = pipeline(
    "image-classification",
    model="Ateeqq/ai-vs-human-image-detector",
    device=0 if torch.cuda.is_available() else -1
)
print("Aigc pipeline successfully loaded!")

# 3. ConvNeXt-V2 AIGC model
print("Loading ConvNeXt AIGC model...")
CONVNEXT_REPO_ID = "xRayon/convnext-ai-images-detector"
CONVNEXT_CKPT_FILENAME = "AI Images Detector/checkpoints/checkpoint_phase2.pth"
convnext_ckpt_path = hf_hub_download(repo_id=CONVNEXT_REPO_ID, filename=CONVNEXT_CKPT_FILENAME)

convnext_model = timm.create_model("convnextv2_base", pretrained=False, num_classes=2)
ckpt = torch.load(convnext_ckpt_path, map_location=device)
convnext_model.load_state_dict(ckpt["model"])
convnext_model.eval()
convnext_model.to(device)
print("ConvNeXt model successfully loaded!")

# ConvNeXt image transform preprocessing pipeline
_convnext_mean = (0.485, 0.456, 0.406)
_convnext_std = (0.229, 0.224, 0.225)
convnext_transform = T.Compose([
    T.Resize(288, interpolation=T.InterpolationMode.LANCZOS),
    T.CenterCrop(256),
    T.ToTensor(),
    T.Normalize(_convnext_mean, _convnext_std),
])

In [ ]:
# Preprocessing & Frame Extraction Helpers
FACE_MARGIN = 40  # margin around faces for GenD

def extract_media_frames(media_path, n_frames=16):
    """
    Uniformly extracts frames from media path. 
    If it's an image, returns list of length 1 containing the image.
    If it's a video, extracts n_frames uniformly.
    """
    if not os.path.exists(media_path):
        raise FileNotFoundError(f"File not found: {media_path}")

    ext = os.path.splitext(media_path)[1].lower()
    if ext in ['.png', '.jpg', '.jpeg', '.webp', '.bmp']:
        # Static image crop
        try:
            img = Image.open(media_path).convert("RGB")
            return [img]
        except Exception as e:
            raise ValueError(f"Could not load image: {e}")
    else:
        # Video extraction
        cap = cv2.VideoCapture(media_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total_frames <= 0:
            # Try reading as PIL image as a fallback
            try:
                img = Image.open(media_path).convert("RGB")
                return [img]
            except Exception:
                raise ValueError(f"Could not extract frames from video: {media_path}")
        
        frame_indices = np.linspace(0, total_frames - 1, num=n_frames, dtype=int)
        frames = []
        
        for idx in frame_indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if not ret:
                break
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(Image.fromarray(frame_rgb))
            
        cap.release()
        return frames

def preprocess_frames_expanded(frames, margin=40):
    """
    Runs YuNet face detector on each frame, cropping the highest-confidence face.
    Falls back to the full frame if no face is detected.
    """
    processed_images = []
    faces_detected = []
    crop_sizes = []
    
    for frame in frames:
        img_bgr = cv2.cvtColor(np.array(frame), cv2.COLOR_RGB2BGR)
        h, w = img_bgr.shape[:2]
        face_detector.setInputSize((w, h))
        _, faces = face_detector.detect(img_bgr)
        
        if faces is not None and len(faces) > 0:
            box = faces[0][:4].astype(int)
            x, y, bw, bh = box
            x1 = max(0, x - margin)
            y1 = max(0, y - margin)
            x2 = min(w, x + bw + margin)
            y2 = min(h, y + bh + margin)
            
            face_crop = frame.crop((x1, y1, x2, y2))
            processed_images.append(face_crop)
            faces_detected.append(True)
            crop_sizes.append((x2 - x1, y2 - y1))
        else:
            processed_images.append(frame)
            faces_detected.append(False)
            crop_sizes.append(None)
            
    return processed_images, faces_detected, crop_sizes

def crop_subtitle_banner_dynamic(pil_image):
    w, h = pil_image.size
    if h > w:  # Vertical
        top_crop = int(h * 0.12)
        bottom_crop = int(h * 0.65)
        return pil_image.crop((0, top_crop, w, bottom_crop))
    else:  # Horizontal/Square
        bottom_crop = int(h * 0.80)
        return pil_image.crop((0, 0, w, bottom_crop))

def extract_photographic_roi_dynamic(pil_image):
    img_bgr = cv2.cvtColor(np.array(pil_image), cv2.COLOR_RGB2BGR)
    h, w = img_bgr.shape[:2]
    
    face_detector.setInputSize((w, h))
    _, faces = face_detector.detect(img_bgr)
    
    is_vertical = h > w
    
    if faces is not None and len(faces) > 0:
        best_face = max(faces, key=lambda f: f[2] * f[3])
        fx, fy, fw, fh = best_face[:4].astype(int)
        
        if is_vertical:
            x1 = max(0, int(fx - fw * 1.0))
            y1 = max(0, int(fy - fh * 0.8))
            x2 = min(w, int(fx + fw * 2.0))
            y2 = min(h, int(fy + fh * 3.5))
        else:
            x1 = max(0, int(fx - fw * 1.2))
            y1 = max(0, int(fy - fh * 0.8))
            x2 = min(w, int(fx + fw * 2.2))
            y2 = min(h, int(fy + fh * 4.5))
        
        return pil_image.crop((x1, y1, x2, y2))
    else:
        if is_vertical:
            x1, y1 = int(w * 0.05), int(h * 0.12)
            x2, y2 = int(w * 0.95), int(h * 0.65)
        else:
            x1, y1 = int(w * 0.15), int(h * 0.10)
            x2, y2 = int(w * 0.85), int(h * 0.75)
        return pil_image.crop((x1, y1, x2, y2))

from PIL import ImageFilter

def apply_unsharp_mask(image):
    return image.filter(ImageFilter.UnsharpMask(radius=2, percent=150, threshold=3))

def strip_infographic_borders(image):
    w, h = image.size
    img_cv = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
    gray = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 50, 150)
    
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    best_crop = None
    max_area = 0
    
    for cnt in contours:
        x, y, cw, ch = cv2.boundingRect(cnt)
        area = cw * ch
        if 0.20 * w * h < area < 0.98 * w * h:
            if area > max_area:
                max_area = area
                best_crop = (x, y, x + cw, y + ch)
                
    if best_crop is not None:
        x1, y1, x2, y2 = best_crop
        crop_w = x2 - x1
        crop_h = y2 - y1
        if crop_w >= 150 and crop_h >= 150:
            return image.crop((x1, y1, x2, y2))
            
    # Fallback to subtitle-cropped full frame
    return image.crop((0, 0, w, int(h * 0.80)))

def normalize_instagram_colors(image):
    img_arr = np.array(image).astype(np.float32) / 255.0
    mean = img_arr.mean(axis=(0, 1))
    std = img_arr.std(axis=(0, 1)) + 1e-6
    img_norm = (img_arr - mean) / std
    
    imagenet_mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    imagenet_std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    img_norm = img_norm * imagenet_std + imagenet_mean
    img_norm = np.clip(img_norm * 255.0, 0, 255).astype(np.uint8)
    return Image.fromarray(img_norm)

def preprocess_media(image, platform="generic"):
    w, h = image.size
    original = image.copy()
    
    if platform == "generic":
        if h > w:
            platform = "tiktok"
        else:
            platform = "facebook"
            
    # 1. ROUTE BY PLATFORM
    if platform == "tiktok":
        image = image.crop((0, int(h * 0.15), int(w * 0.90), int(h * 0.65)))
        image = apply_unsharp_mask(image)
        
    elif platform == "facebook":
        image = strip_infographic_borders(image)
        
    elif platform == "instagram":
        if h > w:
            image = image.crop((0, int(h * 0.12), w, int(h * 0.70)))
        else:
            image = strip_infographic_borders(image)
        image = normalize_instagram_colors(image)
            
    # 2. RESOLUTION SAFEGUARD
    if image.size[0] < 150 or image.size[1] < 150:
        return original.crop((0, 0, w, int(h * 0.80)))
        
    return image


In [ ]:
# Inference, Fusion, and Pipeline Functions
def score_convnext(frame):
    tensor = convnext_transform(frame.convert("RGB")).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = convnext_model(tensor)
        probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]
    return float(probs[1])

def robust_xor_fusion(
    frames,
    faces_detected,
    df_score_by_index,
    aigc_scores,
    df_threshold=0.95,
    ai_threshold=0.85
):
    valid_df_scores = [score for score in df_score_by_index if score is not None]
    
    if valid_df_scores:
        avg_df_prob = float(np.mean(valid_df_scores))
    else:
        avg_df_prob = 0.0

    avg_ai_prob = float(np.mean(aigc_scores)) if aigc_scores else 0.0

    df_fires = avg_df_prob >= df_threshold
    ai_fires = avg_ai_prob >= ai_threshold

    if not df_fires and not ai_fires:
        predicted_label = "Real"
        confidence = 1.0 - max(avg_df_prob, avg_ai_prob)
        explanation = "No face deepfakes or AIGC synthesis detected."
    elif df_fires and (not ai_fires or avg_df_prob >= avg_ai_prob):
        predicted_label = "Deepfake"
        confidence = avg_df_prob
        explanation = "Face manipulation (Deepfake) detected."
    else:
        predicted_label = "AI-Generated"
        confidence = avg_ai_prob
        explanation = "Content is AI-Generated."

    return {
        "predicted_label": predicted_label,
        "confidence": confidence,
        "explanation": explanation,
        "metadata": {
            "avg_df_prob": avg_df_prob,
            "avg_ai_prob": avg_ai_prob,
            "faces_detected": sum(faces_detected),
            "pct_faces": (sum(faces_detected) / len(frames) * 100) if frames else 0.0,
            "n_frames": len(frames),
        }
    }

def analyze_media(
    media_path,
    platform="generic",
    n_frames=16,
    df_threshold=0.95,
    ai_threshold=0.85,
    use_convnext=False
):
    """
    Unified media classification pipeline supporting videos and images.
    """
    raw_frames = extract_media_frames(media_path, n_frames=n_frames)
    if not raw_frames:
        raise ValueError(f"No frames extracted from {media_path}")
        
    # Preprocess frames (strip UI overlays, banners, and apply sharpening)
    frames = [preprocess_media(f, platform) for f in raw_frames]
        
    first_frame = frames[0]
    fw, fh = first_frame.size
    is_vertical = fh > fw
    
    processed_images, faces_detected, crop_sizes = preprocess_frames_expanded(frames, margin=40)

    # 1. Face Deepfake (GenD)
    processed_tensors = [deepfake_model.feature_extractor.preprocess(img) for img in processed_images]
    batch_tensor = torch.stack(processed_tensors).to(device)
    with torch.no_grad():
        df_logits = deepfake_model(batch_tensor)
        df_probs = torch.softmax(df_logits, dim=-1).cpu().numpy()
        
    df_score_by_index = []
    for i, has_face in enumerate(faces_detected):
        if has_face:
            score = float(df_probs[i][1])
            # Texture Guard for Instagram Beauty Filters
            if platform == "instagram":
                face_crop = processed_images[i]
                gray_face = cv2.cvtColor(np.array(face_crop), cv2.COLOR_RGB2GRAY)
                face_var = cv2.Laplacian(gray_face, cv2.CV_64F).var()
                if face_var < 100.0:
                    penalty = max(0.3, face_var / 100.0)
                    score = score * penalty
            df_score_by_index.append(score)
        else:
            df_score_by_index.append(None)

    # 2. AIGC Model on Photographic ROI
    roi_frames = [extract_photographic_roi_dynamic(f) for f in frames]
    if use_convnext:
        aigc_scores = [score_convnext(frame) for frame in roi_frames]
    else:
        aigc_scores = []
        for frame in roi_frames:
            aigc_res = aigc_pipeline(frame)
            aigc_dict = {item['label'].lower(): item['score'] for item in aigc_res}
            aigc_scores.append(aigc_dict.get('ai', 0.0))

    # 3. Dynamic Threshold Adjustment for Vertical Videos / Low-Resolution Crops
    avg_crop_height = np.mean([cs[1] for cs in crop_sizes if cs is not None]) if any(crop_sizes) else 0
    effective_df_threshold = df_threshold
    
    if is_vertical and len(frames) > 1:
        effective_df_threshold = max(effective_df_threshold, 0.98)
    elif 0 < avg_crop_height < 150:
        effective_df_threshold = max(effective_df_threshold, 0.97)

    # 4. XOR Fusion
    result = robust_xor_fusion(
        frames=frames,
        faces_detected=faces_detected,
        df_score_by_index=df_score_by_index,
        aigc_scores=aigc_scores,
        df_threshold=effective_df_threshold,
        ai_threshold=ai_threshold
    )
    result["metadata"]["is_vertical"] = is_vertical
    result["metadata"]["effective_df_threshold"] = effective_df_threshold
    
    return result, frames, roi_frames, faces_detected


In [ ]:
# Traversal and Dataset Parser
def traverse_calibration_data(base_path="../backend/tests/calib_test"):
    """
    Traverses calib_test directory and registers items.
    Format: base_path / <platform> / <category> / <filename>
    """
    base_dir = Path(base_path).resolve()
    
    # Kaggle and environment path auto-detection
    if not base_dir.exists():
        alternative_path = Path("backend/tests/calib_test").resolve()
        kaggle_input = Path("/kaggle/input")
        
        if alternative_path.exists():
            base_dir = alternative_path
            print(f"Using fallback local path: {base_dir}")
        elif kaggle_input.exists():
            # Search for subfolders facebook, instagram, tiktok
            found_path = None
            for p in kaggle_input.glob("**/facebook"):
                found_path = p.parent
                break
            if found_path:
                base_dir = found_path
                print(f"Auto-detected Kaggle dataset path: {base_dir}")
            else:
                print(f"Could not auto-detect calibration folders under /kaggle/input.")
                print("Contents of /kaggle/input:")
                try:
                    for child in kaggle_input.iterdir():
                        print(" - ", child)
                except Exception as e:
                    print("  Error listing /kaggle/input:", e)
                raise FileNotFoundError(f"Calibration test folder not found under /kaggle/input.")
        else:
            raise FileNotFoundError(f"Calibration test folder not found at {base_path} or fallback local path.")

    data_items = []
    supported_extensions = {'.png', '.jpg', '.jpeg', '.webp', '.bmp', '.mp4', '.avi', '.mkv', '.mov'}
    
    for platform_dir in base_dir.iterdir():
        if not platform_dir.is_dir():
            continue
        platform = platform_dir.name
        
        for category_dir in platform_dir.iterdir():
            if not category_dir.is_dir():
                continue
            category = category_dir.name
            
            for file_path in category_dir.iterdir():
                if file_path.is_file() and file_path.suffix.lower() in supported_extensions:
                    data_items.append({
                        "platform": platform,
                        "category": category,
                        "filename": file_path.name,
                        "path": str(file_path)
                    })
                    
    print(f"Discovered {len(data_items)} media files in calibration set.")
    
    df_stats = pd.DataFrame(data_items)
    if not df_stats.empty:
        print("\nDistribution:")
        print(df_stats.groupby(['platform', 'category']).size())
        
    return data_items

In [ ]:
# Evaluation Loop Function
def run_evaluation(data_items, use_convnext=False):
    results = []
    model_name = "ConvNeXt" if use_convnext else "SigLIP"
    print(f"--- Running Evaluation Run with {model_name} ---")
    
    for idx, item in enumerate(data_items):
        path = item["path"]
        print(f"[{idx+1}/{len(data_items)}] {item['platform']}/{item['category']}/{item['filename']}...")
        
        try:
            # Using 8 frames for evaluation run for balanced speed
            res, _, _, _ = analyze_media(path, platform=item['platform'], n_frames=8, use_convnext=use_convnext)
            
            avg_df = res["metadata"]["avg_df_prob"]
            avg_ai = res["metadata"]["avg_ai_prob"]
            verdict = res["predicted_label"]
            conf = res["confidence"]
            
            # Define correctness
            gt = item["category"]
            if gt == "fake":
                is_correct = verdict in ["Deepfake", "AI-Generated"]
            else:
                is_correct = verdict == "Real"
                
            results.append({
                "Platform": item["platform"],
                "Category": item["category"],
                "Filename": item["filename"],
                "DF Avg Score": avg_df,
                "AI Avg Score": avg_ai,
                "Verdict": verdict,
                "Confidence": conf,
                "Correct": is_correct,
                "Path": path
            })
        except Exception as e:
            print(f"  Error processing: {e}")
            results.append({
                "Platform": item["platform"],
                "Category": item["category"],
                "Filename": item["filename"],
                "DF Avg Score": 0.0,
                "AI Avg Score": 0.0,
                "Verdict": f"ERROR: {str(e)[:50]}",
                "Confidence": 0.0,
                "Correct": False,
                "Path": path
            })
            
    return pd.DataFrame(results)

In [ ]:
# Execute Evaluations

# 1. Load the calibration items
data_items = traverse_calibration_data()

# 2. Run SigLIP (Ateeqq) evaluation
df_siglip = run_evaluation(data_items, use_convnext=False)

# 3. Run ConvNeXt evaluation
df_convnext = run_evaluation(data_items, use_convnext=True)

In [ ]:
# Performance and Platform Comparison Reports

print("=========================================================")
print("           MODEL PERFORMANCE METRIC REPORTS")
print("=========================================================")

print(f"\nSigLIP (Ateeqq) Overall Accuracy: {df_siglip['Correct'].mean():.2%}")
print("SigLIP Accuracy by Platform:")
print(df_siglip.groupby("Platform")["Correct"].mean())
print("SigLIP Accuracy by Category:")
print(df_siglip.groupby("Category")["Correct"].mean())

print("\n" + "-"*60)

print(f"\nConvNeXt Overall Accuracy: {df_convnext['Correct'].mean():.2%}")
print("ConvNeXt Accuracy by Platform:")
print(df_convnext.groupby("Platform")["Correct"].mean())
print("ConvNeXt Accuracy by Category:")
print(df_convnext.groupby("Category")["Correct"].mean())

print("=========================================================")

In [ ]:
# Side-by-Side Comparison DataFrame
df_merged = df_siglip[["Platform", "Category", "Filename"]].copy()

df_merged["SigLIP_DF_Score"] = df_siglip["DF Avg Score"]
df_merged["SigLIP_AI_Score"] = df_siglip["AI Avg Score"]
df_merged["SigLIP_Verdict"] = df_siglip["Verdict"]
df_merged["SigLIP_Correct"] = df_siglip["Correct"]

df_merged["ConvNeXt_DF_Score"] = df_convnext["DF Avg Score"]
df_merged["ConvNeXt_AI_Score"] = df_convnext["AI Avg Score"]
df_merged["ConvNeXt_Verdict"] = df_convnext["Verdict"]
df_merged["ConvNeXt_Correct"] = df_convnext["Correct"]

pd.set_option('display.max_rows', len(df_merged))
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("SIDE-BY-SIDE EVALUATION RESULTS:")
df_merged

In [ ]:
# Plotting Accuracy Comparisons
platforms = sorted(list(df_merged["Platform"].unique()))
siglip_platform_accs = [df_siglip[df_siglip["Platform"] == p]["Correct"].mean() for p in platforms]
convnext_platform_accs = [df_convnext[df_convnext["Platform"] == p]["Correct"].mean() for p in platforms]

x = np.arange(len(platforms))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
rects1 = ax.bar(x - width/2, siglip_platform_accs, width, label='SigLIP (Ateeqq)', color='#4c72b0')
rects2 = ax.bar(x + width/2, convnext_platform_accs, width, label='ConvNeXt (xRayon)', color='#55a868')

ax.set_ylabel('Accuracy')
ax.set_title('Pipeline Accuracy by Platform and AIGC Classifier Model')
ax.set_xticks(x)
ax.set_xticklabels(platforms)
ax.set_ylim(0, 1.1)
ax.legend()

def autolabel(rects):
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height:.1%}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3),  # 3 points vertical offset
                    textcoords="offset points",
                    ha='center', va='bottom')

autolabel(rects1)
autolabel(rects2)

plt.tight_layout()
plt.show()

In [ ]:
# Advanced Metrics: FPR (False Positive Rate) & FNR (False Negative Rate)

def compute_fpr_fnr(df):
    # Real items are where Category is 'real' or 'real_text'
    # Fake items are where Category is 'fake'
    real_mask = df["Category"].isin(["real", "real_text"])
    fake_mask = df["Category"] == "fake"
    
    # False Positive: Real item predicted as Deepfake or AI-Generated
    fp = (~df[real_mask]["Correct"]).sum()
    tn = df[real_mask]["Correct"].sum()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    
    # False Negative: Fake item predicted as Real
    fn = (~df[fake_mask]["Correct"]).sum()
    tp = df[fake_mask]["Correct"].sum()
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0
    
    # Specific FPR on 'real_text' to measure subtitle graphic sensitivity
    real_text_mask = df["Category"] == "real_text"
    fp_text = (~df[real_text_mask]["Correct"]).sum()
    tn_text = df[real_text_mask]["Correct"].sum()
    fpr_text = fp_text / (fp_text + tn_text) if (fp_text + tn_text) > 0 else 0.0
    
    # Specific FPR on clean 'real' to isolate template graphic issues
    real_clean_mask = df["Category"] == "real"
    fp_clean = (~df[real_clean_mask]["Correct"]).sum()
    tn_clean = df[real_clean_mask]["Correct"].sum()
    fpr_clean = fp_clean / (fp_clean + tn_clean) if (fp_clean + tn_clean) > 0 else 0.0
    
    return {
        "FPR (Overall)": fpr,
        "FNR (Overall)": fnr,
        "FPR on real_text (Graphics Sensitive)": fpr_text,
        "FPR on clean real (False Alarms)": fpr_clean
    }

siglip_metrics = compute_fpr_fnr(df_siglip)
convnext_metrics = compute_fpr_fnr(df_convnext)

metrics_compare = pd.DataFrame([siglip_metrics, convnext_metrics], index=["SigLIP (Ateeqq)", "ConvNeXt (xRayon)"])

print("=== ADVANCED COMPARATIVE ERROR METRICS ===")
metrics_compare

In [ ]:
# Failure Case Visualizer & Inspector

# Find cases where models disagreed or made errors
print("=== MISCLASSIFIED FILES INSPECTOR ===")
siglip_failures = df_merged[~df_merged["SigLIP_Correct"]].index.tolist()
convnext_failures = df_merged[~df_merged["ConvNeXt_Correct"]].index.tolist()
all_failures = sorted(list(set(siglip_failures + convnext_failures)))

if not all_failures:
    print("Amazing! No failures detected in either model configuration.")
else:
    print(f"Detected {len(all_failures)} total failure cases.")
    print(df_merged.loc[all_failures, ["Platform", "Category", "Filename", "SigLIP_Verdict", "SigLIP_Correct", "ConvNeXt_Verdict", "ConvNeXt_Correct"]])

def inspect_failure_case(idx):
    """
    Helper function to load the frames and visual crops for a specific failure index,
    allowing you to visually see why the model misclassified the file.
    """
    if idx not in df_merged.index:
        print(f"Index {idx} is not in the results.")
        return
        
    row = df_merged.loc[idx]
    path = df_siglip.loc[idx, "Path"]
    print(f"\n--- Inspecting Index {idx} ---")
    print(f"File: {row['Platform']}/{row['Category']}/{row['Filename']}")
    print(f"SigLIP: Verdict={row['SigLIP_Verdict']} (Correct={row['SigLIP_Correct']})")
    print(f"ConvNeXt: Verdict={row['ConvNeXt_Verdict']} (Correct={row['ConvNeXt_Correct']})")
    
    try:
        # Run the full pipeline to get the visual frames for this item
        # (Uses use_convnext=True just to get frames, which are identical for both)
        result, frames, roi_frames, faces_detected = analyze_media(path, n_frames=4, use_convnext=True)
        
        # Print stats
        print(f"Total frames analyzed: {result['metadata']['n_frames']}")
        print(f"Faces detected: {result['metadata']['faces_detected']} frames ({result['metadata']['pct_faces']:.1f}%)")
        print(f"GenD Deepfake Score: {result['metadata']['avg_df_prob']:.2%}")
        print(f"AIGC Classifier Score: {result['metadata']['avg_ai_prob']:.2%}")
        
        # Plot the first frame, its face crop, and its ROI crop
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        
        # 1. Original Frame
        axes[0].imshow(frames[0])
        axes[0].set_title("Original Frame 0")
        axes[0].axis('off')
        
        # 2. Face Detector Crop
        # Run face detection on frame 0 to get crop
        processed_images, faces_det, _ = preprocess_frames_expanded([frames[0]], margin=40)
        axes[1].imshow(processed_images[0])
        face_status = "Face Crop (GenD input)" if faces_det[0] else "No Face (Full Frame Fallback)"
        axes[1].set_title(face_status, color="green" if faces_det[0] else "red")
        axes[1].axis('off')
        
        # 3. Photographic ROI Crop
        axes[2].imshow(roi_frames[0])
        axes[2].set_title("Photographic ROI (AIGC input)")
        axes[2].axis('off')
        
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print("Error loading media details for inspection:", e)

# Example usage: inspect_failure_case(all_failures[0]) if all_failures else None